# D2.7 · Stop authority

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.6 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D2.6.html)**.

| | |
|---|---|
| Tools used | kagent |

## What this lesson is

**What it covers.** Time your own stop authority end to end.

**Why a security engineer needs it.** Nobody has rehearsed halting an autonomous workflow. The control it builds is: named holder, measured time-to-stop, tested.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

At three in the morning, the question is not what went wrong. It is who is allowed to stop it, on what evidence, without waiting for a forty-person bridge call to reach consensus.

> **At CyberTravels.** At three in the morning, who is allowed to stop all four agents without waiting for a bridge call? Pre-agreed authority beats consensus every time, and R1 is what happens while you wait.

## 2 · The framework

```
   03:00, the agent is acting, the evidence is partial

   who may say stop?          +-----------------------------+
                              | named role, on call         |
   on what evidence?          | pre-agreed trigger list     |
   what does stop mean?       | revoke + gateway cut        |
   who is told after?         | named, not assembled at 3am |
                              +-----------------------------+

   pre-agreed authority beats a forty-person bridge call
```

Stop authority is the control everyone assumes exists and almost nobody has
timed.

Five questions decide whether you have it, and each needs a name or a number
rather than an intention:

1. **Who** can halt an agent fleet without seeking approval?
2. **What** is the mechanism — and is it revocation, which survives a restart,
   or process termination, which does not?
3. **How long** does it take, measured end to end, not estimated?
4. **What breaks** when it fires — and has the business already agreed to that?
5. **Who turns it back on**, and against what evidence?

An untested stop button is a belief. The purpose of this lesson is to convert it
into a measurement, because the measurement is what an auditor, a regulator and
a board will each ask for in different words.

## 3 · The procedure, as a skill

The skill prints the vague answers beside the concrete ones, then establishes what each mechanism actually survives — killing the process does not survive a restart, revoking the identity does — and reports a measured twelve-second time-to-stop from a game day rather than an estimate.

In [ ]:
# skills/response/stop-authority-readiness/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: stop-authority-readiness
description: >-
  Turn "we can stop it" into a named mechanism with an owner, a measured
  time-to-stop from a game day, and a stated cost of stopping. Use to answer how
  long it would actually take to stop a running agent, before granting autonomy,
  and whenever stop authority has never been exercised.
allowed-tools: Read, Grep, Glob
---

# Killing the process does not survive a restart

Stop authority is usually described rather than specified: somebody can stop it,
probably quickly, and nobody has tried. Three questions turn that into a control
— which mechanism, who may invoke it without asking, and how long it takes when
measured rather than estimated.

## When to use this

Before raising an agent's autonomy, at any authorisation to run unattended, and
once a year as a game day.

## Procedure

**1 — Write the vague answers down and then the concrete ones.** Side by side.
"Ops can kill it" against "the on-call SRE revokes the workload identity in the
identity console, and here is the runbook". The contrast is what gets the work
scheduled.

**2 — Enumerate mechanisms and what each survives.** Killing the process does not
survive a restart or a scheduler. Revoking the identity does. Blocking egress
stops the effect and not the run. Record what each actually stops.

**3 — Name who may invoke it without asking.** Stop authority that needs an
approval is not stop authority; it is an escalation. If nobody may act alone,
that is the finding.

**4 — Run a game day and measure.** From decision to the agent being unable to
act. Report seconds. An estimate is not a measurement and this is the number
that is always wrong in the optimistic direction.

**5 — Cost the stop.** What stopping costs per minute in halted legitimate work.
Somebody will ask, and having the number is what makes the decision fast during
an incident.

## Output contract

```json
{
  "answers": [{"question": "str", "vague": "str", "concrete": "str"}],
  "mechanisms": [{"name": "str", "stops": ["str"], "survives_restart": false}],
  "authority": {"may_invoke_alone": ["str"], "approval_required": false},
  "game_day": {"measured_seconds": 0, "estimated_seconds": 0},
  "cost_per_minute": 0.0,
  "ready": false
}
```

## Failure modes

- **Process termination as the mechanism.** The scheduler restarts it.
- **An estimated time-to-stop.** Measure it once and the estimate is revealed.
- **Stop authority behind an approval.** That is escalation with a different
  name.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/response/stop-authority-readiness/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/response/stop-authority-readiness/scripts/stop_authority_readiness.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Turn stop authority from a vague answer into a named mechanism with a measured time-to-stop and a known cost.

This is the executable half of the `stop-authority-readiness` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

VAGUE = {
 "who":       "the security team",
 "mechanism": "we can turn off the agents",
 "time":      "quickly",
 "breaks":    "not much",
 "restart":   "when it's safe",
}
CONCRETE = {
 "who":       "on-call SRE, no approval required for non-human identities",
 "mechanism": "revoke the SPIFFE identity at the gateway (survives restart)",
 "time":      "measured 12s decision→first failed call, game day 2026-07-04",
 "breaks":    "auto-remediation pauses; ticket queue grows ~40/hour; "
              "agreed with the service owner 2026-05-11",
 "restart":   "security lead, after the C1.2 containment suite passes on the new build",
}
for k in VAGUE:
    print(f"{k:11s} VAGUE    {VAGUE[k]}")
    print(f"{'':11s} CONCRETE {CONCRETE[k]}\n")

from dataclasses import dataclass

@dataclass
class Agent:
    name: str; running: bool = True; identity_valid: bool = True
    def can_act(self): return self.running and self.identity_valid

MECHANISMS = {
 "kill the process":      (2,   lambda a: setattr(a, "running", False)),
 "network quarantine":    (5,   lambda a: None),
 "revoke the identity":   (12,  lambda a: setattr(a, "identity_valid", False)),
 "rotate the credential": (420, lambda a: setattr(a, "identity_valid", False)),
}
print(f"{'mechanism':24s}{'secs':>6}{'stops it':>10}{'survives restart':>19}")
print("-" * 60)
for name, (secs, apply) in MECHANISMS.items():
    a = Agent("patch-agent")
    apply(a)
    stopped = not a.can_act()
    a.running = True                      # a supervisor restarts the process
    survives = not a.can_act()
    print(f"{name:24s}{secs:>6}{str(stopped):>10}{str(survives):>19}")
print("\nThe fastest mechanism is the one that does not survive a restart.")
print("Speed without persistence is a pause, not a stop.")

GAME_DAY = [
 ("decision made",                    0),
 ("on-call authenticates to the IdP", 4),
 ("identity revoked",                 9),
 ("gateway cache expires",            12),
 ("agent's next call fails",          12),
 ("confirmed in telemetry",           38),
]
print(f"{'step':38s}{'t+s':>6}")
print("-" * 46)
for step, t in GAME_DAY: print(f"{step:38s}{t:>6}")
mttstop = GAME_DAY[4][1]
print(f"\nmeasured time-to-stop: {mttstop}s")
print(f"time-to-confirm:       {GAME_DAY[-1][1]}s")

def cost_of_stop(rate_per_min, seconds):
    return round(rate_per_min * seconds / 60)
for rate in (60, 300, 1200):
    print(f"   at {rate:>5}/min a {mttstop}s stop still permits "
          f"{cost_of_stop(rate, mttstop):>4} further actions")

def stop_authority_ready(answers, measured_seconds, tested_days_ago):
    problems = []
    if any(len(v.split()) < 4 for v in answers.values()):
        problems.append("at least one answer is not specific")
    if measured_seconds is None:
        problems.append("time-to-stop has never been measured")
    if tested_days_ago is None or tested_days_ago > 180:
        problems.append("not tested in the last 180 days")
    return (not problems), problems

for label, ans, secs, days in (("as usually documented", VAGUE, None, None),
                               ("after a game day", CONCRETE, 12, 41)):
    ok, problems = stop_authority_ready(ans, secs, days)
    print(f"\n{label}: ready={ok}")
    for p in problems: print(f"   ⚠ {p}")
assert stop_authority_ready(CONCRETE, 12, 41)[0]

## What you just proved

The vague and concrete answers print side by side. Killing the process stops the agent but does not survive a restart, while identity revocation does. The game-day timeline gives a measured 12-second time-to-stop, permitting 12 to 240 further actions depending on rate. The readiness check fails the vague version on three counts and passes the tested one.

## Your turn

Run the game day. The deliverable is the number, and the number is what goes in the evidence pack for E1.7 and the board slide for E3.5. An untested stop button is a belief.

---

**Next → [D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*